In [14]:
from datasets import load_dataset

dataset = load_dataset("coastalcph/lex_glue", "unfair_tos")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 5532
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1607
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2275
    })
})


In [2]:
#Example
example = dataset["train"][0]
print("Text:", example["text"])
print("Labels:", example["labels"])

Text: notice to california subscribers : you may cancel your subscription , without penalty or obligation , at any time prior to midnight of the third business day following the date you subscribed . 

Labels: []


In [3]:
# What the label numbers correspond to
label_names = dataset["train"].features["labels"].feature.names
print(label_names)

['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']


### Class Distribution

In [4]:
import pandas as pd
from collections import Counter

train_df = dataset["train"].to_pandas()
print(train_df.shape)
train_df.head()

(5532, 2)


,text,labels
0,notice to california subscribers : you may can...,[]
1,"if you subscribed using your apple id , refund...",[]
2,"if you wish to request a refund , please visit...",[]
3,if you subscribed using your google play store...,[]
4,key changes in this version : we 've included ...,[]


In [5]:
label_names = dataset["train"].features["labels"].feature.names

# Count how many times each category appears across all training sentences
flat_labels = [label for labels in train_df["labels"] for label in labels]
label_counts = Counter(flat_labels)

for idx, name in enumerate(label_names):
    print(f"{name}: {label_counts.get(idx, 0)}")

# How many sentences have NO unfair label at all?
no_label_count = sum(1 for labels in train_df["labels"] if len(labels) == 0)
print(f"\nSentences with NO unfair label: {no_label_count} ({no_label_count/len(train_df)*100:.1f}%)")

Limitation of liability: 191
Unilateral termination: 139
Unilateral change: 122
Content removal: 73
Contract by using: 76
Choice of law: 39
Jurisdiction: 34
Arbitration: 28

Sentences with NO unfair label: 4902 (88.6%)


### Sentence Length Distribution

In [6]:
train_df["word_count"] = train_df["text"].apply(lambda x: len(x.split()))

print(train_df["word_count"].describe())

count    5532.000000
mean       32.283080
std        24.745549
min         6.000000
25%        17.000000
50%        26.000000
75%        39.000000
max       441.000000
Name: word_count, dtype: float64


In [7]:
# Bucket sentences into length ranges so we can visualize the distribution
bins = [0, 10, 20, 30, 50, 75, 100, 1000]
bin_labels = ["1-10", "11-20", "21-30", "31-50", "51-75", "76-100", "100+"]

train_df["length_bucket"] = pd.cut(train_df["word_count"], bins=bins, labels=bin_labels)
bucket_counts = train_df["length_bucket"].value_counts().sort_index()

print(bucket_counts)

length_bucket
1-10       497
11-20     1375
21-30     1402
31-50     1447
51-75      543
76-100     151
100+       117
Name: count, dtype: int64


### Label conversion to required format

In [8]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

# Get the validation and test splits as DataFrames too, same way we did for train
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

# Convert label lists into multi-hot binary vectors
mlb = MultiLabelBinarizer(classes=range(len(label_names)))
y_train = mlb.fit_transform(train_df["labels"])
y_val = mlb.transform(val_df["labels"])
y_test = mlb.transform(test_df["labels"])

print(y_train.shape)
print(y_train[:5])

(5532, 8)
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]


### TF-IDF Vectoriztion

In [9]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
X_test = vectorizer.transform(test_df["text"])

print(X_train.shape)

important_terms = ["arbitration", "jurisdiction", "liability", "terminate", "unilateral"]
for term in important_terms:
    status = "in vocabulary" if term in vectorizer.vocabulary_ else "MISSING"
    print(f"{term} → {status}")

(5532, 10000)
arbitration → in vocabulary
jurisdiction → in vocabulary
liability → in vocabulary
terminate → in vocabulary
unilateral → MISSING


In [10]:
matches = train_df[train_df["text"].str.contains("unilateral", case=False)]
print(f"Sentences containing 'unilateral' (any form): {len(matches)}")
print(matches["text"].head(5).tolist())

Sentences containing 'unilateral' (any form): 0
[]


### Model

In [11]:
model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

                         precision    recall  f1-score   support

Limitation of liability       0.53      0.87      0.66        38
 Unilateral termination       0.52      0.84      0.65        38
      Unilateral change       0.52      0.68      0.59        38
        Content removal       0.31      0.92      0.46        13
      Contract by using       0.45      0.78      0.57        23
          Choice of law       0.71      0.92      0.80        13
           Jurisdiction       0.86      0.75      0.80        16
            Arbitration       0.33      0.86      0.48         7

              micro avg       0.50      0.81      0.62       186
              macro avg       0.53      0.83      0.63       186
           weighted avg       0.53      0.81      0.63       186
            samples avg       0.08      0.09      0.08       186



In [12]:
model_unweighted = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)
model_unweighted.fit(X_train, y_train)
y_pred_unweighted = model_unweighted.predict(X_test)

print("=== WITHOUT class_weight='balanced' ===")
print(classification_report(y_test, y_pred_unweighted, target_names=label_names, zero_division=0))

=== WITHOUT class_weight='balanced' ===
                         precision    recall  f1-score   support

Limitation of liability       0.83      0.26      0.40        38
 Unilateral termination       1.00      0.18      0.31        38
      Unilateral change       1.00      0.16      0.27        38
        Content removal       1.00      0.15      0.27        13
      Contract by using       1.00      0.04      0.08        23
          Choice of law       0.00      0.00      0.00        13
           Jurisdiction       0.00      0.00      0.00        16
            Arbitration       0.00      0.00      0.00         7

              micro avg       0.93      0.14      0.24       186
              macro avg       0.60      0.10      0.17       186
           weighted avg       0.77      0.14      0.23       186
            samples avg       0.02      0.02      0.02       186

